# NodeDiffusion 注意力结构消融评估
- **评估**：从训练集随机抽取 500 条，DDPM 1000 步推理，对比各变体 Coord-RMSE 与方差
- **可视化**：从 500 条中随机抽取 5 条，生成论文级排版对比图

In [ ]:
import os, shutil
REPO_DIR = '/kaggle/working/PlanDiffusion_wzm'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.system(f'git clone https://github.com/WeeZHnMin/PlanDiffusion_wzm.git {REPO_DIR}')
print(f'repo ready: {REPO_DIR}')

In [ ]:
import os

BERT_PATH = 'bert-base-uncased'
DATA_PATH = '/kaggle/input/datasets/yahiie/node-diffusion-6k/graph_dataset_6k.npz'
CKPT_DIR  = '/kaggle/working/checkpoints_eval'
OUT_DIR   = '/kaggle/working/ablation_out'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

HF_REPOS = {
    'adj_only':    'wzmmmm/plandiff-adj-cross-6k',
    'global_only': 'wzmmmm/plandiff-global-cross-6k',
    'dual_stream': 'wzmmmm/plandiff-double-cross-6k',
}

MODEL_CHANNELS = 384
NUM_LAYERS     = 6
NUM_HEADS      = 6
TIMESTEPS      = 1000
N_EVAL         = 500
N_VIZ          = 5
SEED           = 42

print('config OK')
hf_ok = ('OK (' + HF_TOKEN[:8] + '...)') if HF_TOKEN else '未设置'
print(f'HF_TOKEN: {hf_ok}')

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

def timestep_embedding(timesteps, dim):
    half  = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, dtype=torch.float32, device=timesteps.device) / half
    )
    args = timesteps[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

def attention(q, k, v, d_k, mask=None, dropout=None):
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask.unsqueeze(1) == 1, -1e4)
    scores = F.softmax(scores.float(), dim=-1).to(q.dtype)
    if dropout is not None:
        scores = dropout(scores)
    return torch.matmul(scores, v)

class MultiHeadAttention(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()
        self.d_k = d_model // heads; self.h = heads
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out      = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        q  = self.q_linear(q).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        k  = self.k_linear(k).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        v  = self.v_linear(v).view(bs, -1, self.h, self.d_k).transpose(1, 2)
        out = attention(q, k, v, self.d_k, mask, self.dropout)
        return self.out(out.transpose(1, 2).contiguous().view(bs, -1, self.h * self.d_k))

class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model * 2, d_model)
        )
    def forward(self, x): return self.net(x)

class EncoderLayer_Dual(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm_x = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.adj_attn = MultiHeadAttention(heads, d_model, dropout)
        self.glb_attn = MultiHeadAttention(heads, d_model, dropout)
        self.crs_attn = MultiHeadAttention(heads, d_model, dropout)
        self.ff = FeedForward(d_model, dropout); self.drop = nn.Dropout(dropout)
    def forward(self, x, adj_mask, tf, tm):
        x2 = self.norm1(x)
        x  = x + self.drop(self.adj_attn(x2, x2, x2, adj_mask) + self.glb_attn(x2, x2, x2, None))
        x2 = self.norm_x(x); x = x + self.drop(self.crs_attn(x2, tf, tf, tm))
        x2 = self.norm2(x);  x = x + self.drop(self.ff(x2))
        return x

class EncoderLayer_Adj(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm_x = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.adj_attn = MultiHeadAttention(heads, d_model, dropout)
        self.crs_attn = MultiHeadAttention(heads, d_model, dropout)
        self.ff = FeedForward(d_model, dropout); self.drop = nn.Dropout(dropout)
    def forward(self, x, adj_mask, tf, tm):
        x2 = self.norm1(x); x = x + self.drop(self.adj_attn(x2, x2, x2, adj_mask))
        x2 = self.norm_x(x); x = x + self.drop(self.crs_attn(x2, tf, tf, tm))
        x2 = self.norm2(x);  x = x + self.drop(self.ff(x2))
        return x

class EncoderLayer_Global(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm_x = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.glb_attn = MultiHeadAttention(heads, d_model, dropout)
        self.crs_attn = MultiHeadAttention(heads, d_model, dropout)
        self.ff = FeedForward(d_model, dropout); self.drop = nn.Dropout(dropout)
    def forward(self, x, adj_mask, tf, tm):
        x2 = self.norm1(x); x = x + self.drop(self.glb_attn(x2, x2, x2, None))
        x2 = self.norm_x(x); x = x + self.drop(self.crs_attn(x2, tf, tf, tm))
        x2 = self.norm2(x);  x = x + self.drop(self.ff(x2))
        return x

class NodeDiffusionTransformer(nn.Module):
    def __init__(self, layer_cls, model_channels=384, num_layers=6, num_heads=6,
                 dropout=0.1, bert_name='bert-base-uncased'):
        super().__init__()
        self.mc = model_channels
        self.time_embed = nn.Sequential(
            nn.Linear(model_channels, model_channels), nn.SiLU(),
            nn.Linear(model_channels, model_channels))
        self.input_emb  = nn.Linear(2, model_channels)
        self.bert = BertModel.from_pretrained(bert_name)
        for p in self.bert.parameters(): p.requires_grad = False
        self.text_proj  = nn.Linear(self.bert.config.hidden_size, model_channels)
        self.layers     = nn.ModuleList([layer_cls(model_channels, num_heads, dropout) for _ in range(num_layers)])
        self.coord_head = nn.Sequential(
            nn.Linear(model_channels, model_channels), nn.ReLU(),
            nn.Linear(model_channels, model_channels // 2),
            nn.Linear(model_channels // 2, 2))
    def _adj_mask(self, adj, mask):
        return torch.clamp((1 - adj) + (1 - mask).unsqueeze(1), 0, 1)
    def forward(self, x, timesteps, adj_matrix, node_mask,
                prompt_tokens=None, prompt_mask=None, **kw):
        B, _, N = x.shape
        x  = x.permute(0, 2, 1).float()
        te = self.time_embed(timestep_embedding(timesteps, self.mc)).unsqueeze(1)
        h  = self.input_emb(x) + te
        am = self._adj_mask(adj_matrix.float(), node_mask.float())
        if prompt_tokens is not None:
            ba = prompt_mask if prompt_mask is not None else (prompt_tokens != 0).long()
            with torch.no_grad():
                th = self.bert(input_ids=prompt_tokens, attention_mask=ba).last_hidden_state
            tf = self.text_proj(th)
            tm = (1 - ba.float()).unsqueeze(1)
        else:
            tf = torch.zeros(B, 1, self.mc, device=h.device, dtype=h.dtype)
            tm = None
        for layer in self.layers:
            h = layer(h, am, tf, tm)
        return self.coord_head(h).permute(0, 2, 1)

print('模型类定义完成')

In [ ]:
import math, torch

class GaussianDiffusion:
    def __init__(self, timesteps=1000):
        self.T = timesteps
        t  = torch.arange(timesteps + 1) / timesteps
        f  = torch.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2
        ab = f / f[0]
        b  = (1 - ab[1:] / ab[:-1]).clamp(max=0.999)
        ab = ab[1:]
        a  = 1.0 - b
        ap = torch.cat([torch.tensor([1.0]), ab[:-1]])
        self.betas = b; self.alphas = a
        self.alphas_bar = ab; self.alphas_bar_prev = ap
        self.post_var = (b * (1 - ap) / (1 - ab)).clamp(min=1e-20)
    def _to(self, dev):
        for attr in ['betas','alphas','alphas_bar','alphas_bar_prev','post_var']:
            setattr(self, attr, getattr(self, attr).to(dev))
        return self

@torch.no_grad()
def ddpm_sample(model, diff, cond, device):
    diff._to(device)
    x = torch.randn(1, 2, 40, device=device)
    for t in reversed(range(diff.T)):
        tb  = torch.tensor([t], device=device, dtype=torch.long)
        eps = model(x, tb, **cond).float()
        ab  = diff.alphas_bar[t]; ap = diff.alphas_bar_prev[t]
        a   = diff.alphas[t];     b  = diff.betas[t]
        x0  = ((x - (1 - ab).sqrt() * eps) / ab.sqrt().clamp(min=1e-3)).clamp(-300, 300)
        mu  = (ap.sqrt() * b / (1 - ab)) * x0 + (a.sqrt() * (1 - ap) / (1 - ab)) * x
        x   = mu + diff.post_var[t].sqrt() * torch.randn_like(x) if t > 0 else mu
    return x[0].permute(1, 0).cpu().numpy()  # [40, 2]

diffusion = GaussianDiffusion(timesteps=TIMESTEPS)
print('GaussianDiffusion + DDPM sampler OK')

In [ ]:
from huggingface_hub import hf_hub_download
import torch

LAYER_MAP = {
    'adj_only':    EncoderLayer_Adj,
    'global_only': EncoderLayer_Global,
    'dual_stream': EncoderLayer_Dual,
}

models = {}
for name, repo_id in HF_REPOS.items():
    print(f'loading {name} <- {repo_id} ...')
    try:
        path = hf_hub_download(repo_id=repo_id, filename='latest.pt',
                               token=HF_TOKEN,
                               local_dir=os.path.join(CKPT_DIR, name),
                               force_download=True)
    except Exception as e:
        print(f'  [skip] {name}: {e}'); continue
    m = NodeDiffusionTransformer(
        layer_cls=LAYER_MAP[name], model_channels=MODEL_CHANNELS,
        num_layers=NUM_LAYERS, num_heads=NUM_HEADS, bert_name=BERT_PATH,
    ).to(device)
    ckpt  = torch.load(path, map_location=device)
    state = ckpt['model']
    if any(k.startswith('module.') for k in state):
        state = {k[7:]: v for k, v in state.items()}
    m.load_state_dict(state, strict=False)
    m.eval()
    models[name] = m
    print(f'  {name}: step={ckpt.get("step","?")}')

if not models:
    raise RuntimeError('no model loaded')
print(f'\nLoaded: {list(models.keys())}')

In [ ]:
import numpy as np, torch

torch.manual_seed(SEED); np.random.seed(SEED)

data  = np.load(DATA_PATH, allow_pickle=True)
total = len(data['node_coords'])
rng   = np.random.default_rng(SEED)
idxs  = sorted(rng.choice(total, size=N_EVAL, replace=False).tolist())
print(f'Dataset: {total}  Eval: {N_EVAL}')

rmse_list = {n: [] for n in models}
pred_cache = {n: [] for n in models}
gt_cache, mask_cache, adj_cache, type_cache = [], [], [], []

for i, idx in enumerate(idxs):
    gt   = data['node_coords'][idx].astype('float32')
    adj  = data['adj_matrix'][idx].astype('float32')
    mask = data['node_mask'][idx].astype('float32')
    typ  = data['node_combo_ids'][idx].astype('int64')
    ptok = data['prompt_tokens'][idx].astype('int64')
    pmsk = data['prompt_mask'][idx].astype('float32')
    cond = {
        'adj_matrix':    torch.from_numpy(adj[None]).to(device),
        'node_mask':     torch.from_numpy(mask[None]).to(device),
        'prompt_tokens': torch.from_numpy(ptok[None]).to(device),
        'prompt_mask':   torch.from_numpy(pmsk[None]).to(device),
    }
    valid = mask > 0.5
    gt_cache.append(gt); adj_cache.append(adj)
    mask_cache.append(valid); type_cache.append(typ)
    for name, model in models.items():
        pred = ddpm_sample(model, diffusion, cond, device)
        rmse = float(np.sqrt(np.mean((pred[valid] - gt[valid]) ** 2)))
        rmse_list[name].append(rmse)
        pred_cache[name].append(pred)
    if (i + 1) % 50 == 0:
        row = ' | '.join(f'{n}: {np.mean(rmse_list[n]):.2f}' for n in models)
        print(f'[{i+1}/{N_EVAL}] {row}')

DISPLAY_NAME = {
    'adj_only':    'Adj-only',
    'global_only': 'Global-only',
    'dual_stream': 'Dual-stream',
}

print()
print(f'{"Variant":<14} {"Mean":>7} {"Std":>7} {"Var":>8} {"Median":>8} {"Min":>7} {"Max":>7}')
print('-' * 60)
for name in models:
    r = np.array(rmse_list[name])
    print(f'{DISPLAY_NAME[name]:<14} {r.mean():>7.2f} {r.std():>7.2f} {r.var():>8.2f} {np.median(r):>8.2f} {r.min():>7.2f} {r.max():>7.2f}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
from IPython.display import display, Image as IPImage

# ── paper style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Serif',
    'font.size':         7,
    'axes.titlesize':    7,
    'axes.titleweight':  'bold',
    'axes.linewidth':    0.6,
    'xtick.major.size':  0,
    'ytick.major.size':  0,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.03,
})

NODE_PALETTE = [
    '#4E8CC2','#7BB9E0','#B0D4F0','#1A5F8A',
    '#4DA863','#82C78A','#B5E0B8','#1A6B2A',
    '#D4A520','#E8C060','#F5DC99','#9A7010',
    '#9B5DB5','#C090D8','#E0C0F0','#6A2E8A',
    '#D04040','#E88080','#F5B5B5','#8A1A1A',
    '#808080','#A8A8A8','#C8C8C8','#585858',
    '#D07830','#E8A870','#F5CCA8','#8A4A10',
    '#30A0A0','#70C8C8','#A8E0E0','#107070',
]

def node_color(tid):
    idx = (int(tid) - 1) % len(NODE_PALETTE) if 1 <= int(tid) <= 32 else -1
    return NODE_PALETTE[idx] if idx >= 0 else '#CCCCCC'

def draw_cell(ax, xy, types, adj, n, xlim, ylim, rmse=None):
    # edges
    for ii in range(n):
        for jj in range(ii + 1, n):
            if adj[ii, jj] > 0.5:
                ax.plot([xy[ii,0], xy[jj,0]], [xy[ii,1], xy[jj,1]],
                        color='#BBBBBB', lw=0.7, zorder=1, solid_capstyle='round')
    # nodes
    for k in range(n):
        c = node_color(types[k])
        ax.scatter(xy[k,0], xy[k,1], color=c, s=28, zorder=3,
                   edgecolors='#444444', linewidths=0.4)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect('equal'); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_linewidth(0.5); sp.set_color('#AAAAAA')
    if rmse is not None:
        ax.text(0.5, -0.06, f'RMSE = {rmse:.1f} px',
                transform=ax.transAxes, ha='center', va='top',
                fontsize=6, color='#333333')

# ── sample selection ──────────────────────────────────────────────────────────
rng_viz  = np.random.default_rng(SEED + 99)
viz_idxs = sorted(rng_viz.choice(N_EVAL, size=N_VIZ, replace=False).tolist())

var_keys = list(models.keys())
ROW_LABELS = {
    'gt':          'Ground Truth',
    'adj_only':    'Adj-only',
    'global_only': 'Global-only',
    'dual_stream': 'Dual-stream',
}
row_keys = ['gt'] + var_keys
n_rows   = len(row_keys)
n_cols   = N_VIZ

# ── layout ────────────────────────────────────────────────────────────────────
cell_w, cell_h = 2.4, 2.4
label_w        = 1.0
fig_w = label_w + cell_w * n_cols
fig_h = cell_h * n_rows

fig = plt.figure(figsize=(fig_w, fig_h))

# column positions (normalised)
col_starts = [(label_w + cell_w * c) / fig_w for c in range(n_cols)]
col_width  = cell_w / fig_w
row_starts = [1.0 - cell_h * (r + 1) / fig_h for r in range(n_rows)]
row_height = cell_h / fig_h
margin     = 0.012   # gap between cells

axes = {}
for ri, rk in enumerate(row_keys):
    for ci, si in enumerate(viz_idxs):
        left   = col_starts[ci] + margin
        bottom = row_starts[ri] + margin
        w      = col_width  - 2 * margin
        h      = row_height - 2 * margin
        ax = fig.add_axes([left, bottom, w, h])
        axes[(ri, ci)] = ax

# ── draw ──────────────────────────────────────────────────────────────────────
for ci, si in enumerate(viz_idxs):
    idx    = idxs[si]
    n_node = int(mask_cache[si].sum())
    gt_xy  = gt_cache[si]
    adj_np = adj_cache[si]
    types  = type_cache[si]

    # compute common axis limits from GT (same scale for all rows in this col)
    pts  = gt_xy[:n_node]
    pad  = max(12.0, 0.08 * float(np.ptp(pts, axis=0).max()))
    xlim = (pts[:,0].min() - pad, pts[:,0].max() + pad)
    ylim = (pts[:,1].min() - pad, pts[:,1].max() + pad)

    # col header: sample index
    ax0 = axes[(0, ci)]
    ax0.set_title(f'Sample {ci+1}', pad=3, fontsize=7, fontweight='bold', color='#222222')

    for ri, rk in enumerate(row_keys):
        ax = axes[(ri, ci)]
        if rk == 'gt':
            draw_cell(ax, gt_xy, types, adj_np, n_node, xlim, ylim)
        else:
            rmse_v = rmse_list[rk][si]
            draw_cell(ax, pred_cache[rk][si], types, adj_np, n_node, xlim, ylim, rmse=rmse_v)

# ── row labels (left margin) ──────────────────────────────────────────────────
for ri, rk in enumerate(row_keys):
    x_norm = (label_w * 0.5) / fig_w
    y_norm = row_starts[ri] + row_height * 0.5
    fig.text(x_norm, y_norm, ROW_LABELS.get(rk, rk),
             ha='center', va='center', fontsize=7, fontweight='bold',
             color='#111111', rotation=90)

# ── divider line between GT and predictions ───────────────────────────────────
div_y = (row_starts[0] - margin * 0.5)
fig.add_artist(plt.Line2D([label_w / fig_w, 1.0], [div_y, div_y],
                           transform=fig.transFigure,
                           color='#AAAAAA', lw=0.6, linestyle='--'))

out_path = os.path.join(OUT_DIR, 'ablation_viz.pdf')
fig.savefig(out_path)
out_png  = os.path.join(OUT_DIR, 'ablation_viz.png')
fig.savefig(out_png, dpi=300)
plt.close(fig)
display(IPImage(out_png))
print(f'saved: {out_path}')
print(f'saved: {out_png}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image as IPImage

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 8,
    'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 300, 'savefig.dpi': 300,
})

COLORS = {'adj_only': '#4E8CC2', 'global_only': '#4DA863', 'dual_stream': '#D04040'}
LABELS = {'adj_only': 'Adj-only', 'global_only': 'Global-only', 'dual_stream': 'Dual-stream'}

var_keys = list(models.keys())
means    = [np.mean(rmse_list[k]) for k in var_keys]
stds     = [np.std(rmse_list[k])  for k in var_keys]
colors   = [COLORS.get(k, '#888888') for k in var_keys]
labels   = [LABELS.get(k, k) for k in var_keys]

fig, ax = plt.subplots(figsize=(3.2, 2.6))
x = np.arange(len(var_keys))
bars = ax.bar(x, means, yerr=stds, width=0.5,
              color=colors, alpha=0.85,
              error_kw=dict(elinewidth=0.8, capsize=3, capthick=0.8, ecolor='#444444'))
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel('Coord-RMSE (px)', fontsize=8)
ax.set_xlabel('Attention Variant', fontsize=8)
ax.yaxis.grid(True, linewidth=0.4, linestyle='--', color='#CCCCCC', zorder=0)
ax.set_axisbelow(True)

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            mean + std + 0.5,
            f'{mean:.1f}±{std:.1f}',
            ha='center', va='bottom', fontsize=6, color='#222222')

fig.tight_layout()
out_bar = os.path.join(OUT_DIR, 'ablation_bar.pdf')
fig.savefig(out_bar)
out_bar_png = os.path.join(OUT_DIR, 'ablation_bar.png')
fig.savefig(out_bar_png)
plt.close(fig)
display(IPImage(out_bar_png))
print(f'saved: {out_bar}')